# 🧰 Notebook 3 — Event Design in the Real World

> **Goal:** Cover the four things that separate a toy demo from a production event-driven system:
> 1. **Events vs commands** — naming your messages right.
> 2. **Schema evolution** — adding fields without breaking consumers.
> 3. **Delivery semantics & idempotency** — surviving "at-least-once" delivery.
> 4. **Broker vs Mediator topology** — where the routing logic lives.

Every section runs in pure Python — no Kafka, no Rabbit. The patterns are exactly the same when you move to a real broker.


## 🛠️ Setup

```bash
cd 05-microservices/event-driven-architecture
uv sync
```

Then in VS Code, select the `.venv` kernel (top-right of the notebook).
If it isn't listed, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 1️⃣ Events (facts) vs Commands

A subtle but high-leverage distinction.

| Type        | Intent                         | Past or future?        | Example                  | Who decides?       |
|-------------|--------------------------------|------------------------|--------------------------|--------------------|
| **Command** | "Please do X"                  | Future (imperative)    | `ShipOrder`, `ChargeCard`| **Producer**        |
| **Event**   | "X happened" (fact)            | Past (indicative)      | `OrderPlaced`, `CardCharged` | **Consumer(s)** |

### Rule of thumb
- **Commands** have **exactly one** intended handler. You're telling a specific service what to do.
- **Events** can have **zero or many** handlers. You're announcing that something happened; whoever cares can react.

Naming:
- ❌ `order_placed` emitted to tell shipping to ship → this is a hidden command.
- ✅ `OrderPlaced` emitted as a fact. Shipping, email, loyalty, analytics *all* subscribe independently.

Let's build both and feel the difference.


In [ ]:
from collections import defaultdict

class Bus:
    def __init__(self): self.subs = defaultdict(list)
    def subscribe(self, topic, fn): self.subs[topic].append(fn)
    def publish(self, topic, payload):
        for fn in self.subs[topic]:
            fn(payload)

bus = Bus()

# ---- COMMAND: point-to-point, exactly one handler expected ----
def ship_handler(cmd):
    print(f"  📦 SHIP executed for order={cmd['order_id']}")
bus.subscribe("ShipOrder", ship_handler)

# ---- EVENT (fact): pub/sub, many handlers ----
bus.subscribe("OrderPlaced", lambda e: print(f"  ✉️  email   order={e['order_id']}"))
bus.subscribe("OrderPlaced", lambda e: print(f"  📊 metrics order={e['order_id']}"))
bus.subscribe("OrderPlaced", lambda e: print(f"  ⭐ loyalty order={e['order_id']}"))

print("— Sending a COMMAND (1 handler):")
bus.publish("ShipOrder",   {"order_id": 42})

print("\n— Publishing an EVENT (many handlers):")
bus.publish("OrderPlaced", {"order_id": 42})

> **Why this matters:** if you confuse the two, your system drifts into a distributed monolith — producers that *"publish an event"* but secretly depend on which consumers run, in which order.


## 2️⃣ Schema evolution — add fields without breaking consumers

Your event is a **public contract**. Once it's out in the wild, old consumers will still be running the v1 shape while new consumers expect v2. The safe rules:

- ✅ **Add optional fields.** Old consumers ignore them.
- ✅ **Include a `version` (or schema URL).**
- ❌ **Never rename or remove** a field that old consumers read.
- ❌ **Never change the meaning** of an existing field.

If you *must* break compatibility, publish a new event type (`OrderPlacedV2`) side-by-side and migrate consumers one at a time.


In [ ]:
# v1 event: order_id + total
# v2 event: adds 'currency' (optional). v1 consumers must still work.

def legacy_receipt(e):
    # Written when only v1 existed. Uses .get so missing fields are safe.
    print(f"  📨 [legacy] order={e['order_id']} total={e['total']}")

def modern_receipt(e):
    # Aware of v2. Defaults currency for old events.
    cur = e.get("currency", "USD")
    print(f"  📨 [modern] order={e['order_id']} total={e['total']} {cur}")

bus = Bus()
bus.subscribe("OrderPlaced", legacy_receipt)
bus.subscribe("OrderPlaced", modern_receipt)

print("— v1 event (no 'currency'):")
bus.publish("OrderPlaced", {"version": 1, "order_id": 1, "total": 42})

print("\n— v2 event (adds 'currency'):")
bus.publish("OrderPlaced", {"version": 2, "order_id": 2, "total": 42, "currency": "EUR"})

Both consumers survive both versions. This is the magic of **additive, optional-only** schema changes.

> **Tip:** validate events against a schema (JSON Schema, Protobuf, Avro) *at the producer* before publishing. Catching a typo in CI beats debugging it in production at 3am.


## 3️⃣ Delivery semantics & idempotency

Most real brokers guarantee **at-least-once** delivery. That's a pragmatic choice: exactly-once is expensive and often an illusion. In practice this means:

> The same event can — and occasionally will — arrive twice.

Your consumer must be **idempotent**: processing the same event N times produces the same effect as processing it once.

Easiest way: put a unique `event_id` on every event and make consumers remember the IDs they've processed.


In [ ]:
import uuid

bus = Bus()
evt = {"event_id": str(uuid.uuid4()), "order_id": 99, "amount": 50}

# ---------- ❌ naive consumer: no dedupe ----------
ledger_naive = {"charged": 0}

def charge_card_naive(e):
    ledger_naive["charged"] += e["amount"]
    print(f"  💳 charging ${e['amount']} for order={e['order_id']}")

bus.subscribe("OrderPlaced", charge_card_naive)

print("— naive consumer, broker delivers the SAME event twice:")
bus.publish("OrderPlaced", evt)
bus.publish("OrderPlaced", evt)          # at-least-once: this WILL happen to you
print(f"  total charged: ${ledger_naive['charged']}  ← customer paid twice for one order 💥")

# ---------- ✅ idempotent consumer: dedupe on event_id ----------
processed_ids = set()
ledger = {"charged": 0}

def charge_card(e):
    if e["event_id"] in processed_ids:
        print(f"  🛡️  duplicate {e['event_id'][:8]} — skipping")
        return
    processed_ids.add(e["event_id"])
    ledger["charged"] += e["amount"]
    print(f"  💳 charging ${e['amount']} for order={e['order_id']} "
          f"(event {e['event_id'][:8]})")

bus = Bus()
bus.subscribe("OrderPlaced", charge_card)

print("\n— idempotent consumer, same duplicate delivery:")
bus.publish("OrderPlaced", evt)
bus.publish("OrderPlaced", evt)
print("— a genuinely different order (new event_id):")
bus.publish("OrderPlaced", {"event_id": str(uuid.uuid4()), "order_id": 100, "amount": 75})
print(f"  total charged: ${ledger['charged']}  ← correct ✅")


> ⚠️ **The in-memory `set` above is a teaching prop, not a design.** It dies with the
> process, so a restart re-opens the double-charge window, and two consumer replicas
> each keep their own copy. In production the dedupe key must be **durable and shared**:
> a `UNIQUE` constraint on `(consumer_name, event_id)` in the consumer's own database,
> written **in the same transaction as the side effect**. If they aren't in one
> transaction you have just recreated the dual-write problem one layer down.
>
> Better still, make the effect **naturally idempotent** where you can —
> `UPDATE ... SET status='paid' WHERE id=? AND status='pending'` needs no dedupe table
> at all.

### Other delivery hazards to know about
- **Out-of-order delivery.** Rarely are events strictly ordered across a whole topic.
  Use a partition key (the entity id) so events for one entity stay ordered, and carry a
  per-entity sequence number so consumers can drop anything they've already passed.
- **Poison messages.** A malformed event that keeps crashing the consumer. Real brokers
  ship it to a **Dead Letter Queue (DLQ)** after N retries — see notebook 4.
- **Backpressure.** If consumers fall behind, queues grow. Monitor queue lag (Kafka
  consumer lag, SQS `ApproximateAgeOfOldestMessage`).

## 4️⃣ Topology: Broker vs Mediator

Two canonical ways to wire producers and consumers.

### 🕸️ Broker topology (decentralised)

- Producers publish events to topics. Consumers subscribe to topics. **No central brain.**
- Each consumer decides what to do and may publish follow-up events.
- 👍 Simple, scales horizontally, highly decoupled.
- 👎 The overall business flow is **implicit** — you read it by tracing topics across services.

### 🧠 Mediator topology (centralised)

- A **mediator/orchestrator** receives an initial event and explicitly dispatches sub-commands in order, handles failures, knows the whole workflow.
- 👍 The workflow is **explicit**; easier to reason about, audit, and add compensation logic. (This is how Sagas often look.)
- 👎 The mediator can become a bottleneck and a deployment coupling point.

Let's implement the same "place an order" flow in both styles and contrast them.


In [ ]:
# ---------- Broker style ----------
# Each service reacts to events and emits its own. No central coordinator.

broker = Bus()

def inventory(e):
    print(f"  📦 inventory reserved for order={e['order_id']}")
    broker.publish("InventoryReserved", {"order_id": e["order_id"]})

def payment(e):
    print(f"  💳 payment captured   for order={e['order_id']}")
    broker.publish("PaymentCaptured",  {"order_id": e["order_id"]})

def shipper(e):
    print(f"  🚚 shipping label     for order={e['order_id']}")

broker.subscribe("OrderPlaced",        inventory)
broker.subscribe("InventoryReserved",  payment)
broker.subscribe("PaymentCaptured",    shipper)

print("— Broker topology:")
broker.publish("OrderPlaced", {"order_id": 1})

In [ ]:
# ---------- Mediator style ----------
# One component owns the workflow. Services expose commands; the mediator
# decides the sequence and is the single place that knows the happy path.

def inventory_cmd(order_id):
    print(f"  📦 inventory reserved for order={order_id}")
    return True

def payment_cmd(order_id):
    print(f"  💳 payment captured   for order={order_id}")
    return True

def shipping_cmd(order_id):
    print(f"  🚚 shipping label     for order={order_id}")
    return True

class OrderOrchestrator:
    def handle_order_placed(self, order_id):
        if not inventory_cmd(order_id):
            print("  ❌ inventory failed — aborting"); return
        if not payment_cmd(order_id):
            print("  ↩️  compensating: release inventory"); return
        shipping_cmd(order_id)

print("— Mediator topology:")
OrderOrchestrator().handle_order_placed(2)

### When to pick which

| Situation                                 | Prefer    |
|-------------------------------------------|-----------|
| Simple fan-out / notifications            | Broker    |
| Independent teams shipping independently  | Broker    |
| Complex multi-step workflow with compensation (Saga) | Mediator |
| You need one place to answer "where is order 123?" | Mediator |

In practice most large systems end up **hybrid**: brokered pub/sub for cross-cutting concerns (analytics, notifications) *plus* an orchestrator for the few critical business workflows (checkout, onboarding).


## 🎁 Mapping to real technologies

| Concept here           | In production you'll see…                                |
|------------------------|----------------------------------------------------------|
| `EventBus`             | Kafka, RabbitMQ, NATS, AWS SNS+SQS, Google Pub/Sub, Redis Streams |
| `topic` / event name   | Kafka topic, RabbitMQ exchange, SNS topic                |
| `subscribe`            | Kafka consumer group, SQS queue, Rabbit queue binding    |
| `event_id` dedup       | Idempotency key, outbox pattern, consumer offsets        |
| Mediator orchestrator  | Temporal, AWS Step Functions, Camunda, Netflix Conductor |
| DLQ                    | Kafka DLT, SQS redrive policy, RabbitMQ DLX              |

## 🚦 When should *you* reach for EDA?

✅ Good fit
- Many independent reactions to the same business fact (orders, signups, payments).
- Wildly different traffic profiles across consumers.
- You want to add features without re-deploying the producers.

🚫 Poor fit
- Simple CRUD apps with one database and one team.
- Strong, synchronous consistency needs (an airline booking seat inventory).
- Tiny systems where a function call is genuinely enough. Don't cargo-cult it.

## 🧪 Try it yourself
1. Extend section 3 with a second idempotent consumer (e.g., `send_receipt`).
2. In section 4, simulate a payment failure in the mediator and add compensation for inventory.
3. Swap our in-memory `Bus` for `redis.Redis().publish` / `pubsub.subscribe` — the patterns don't change.
